# STAI-X 2026 — EDA

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from src.data_loader import load_train, load_val

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.3f}'.format)

In [ ]:
train = load_train()
val   = load_val()
print('Train:', train.shape, '  Val:', val.shape)
train.head(3)

## 1. Target suppression rates

In [ ]:
supp = train.groupby('overdose_category')['is_suppressed'].mean().rename('suppression_rate')
print(supp.to_string())

# Suppression by jurisdiction (top 15)
by_juris = train.groupby('jurisdiction')['is_suppressed'].mean().sort_values(ascending=False).head(15)
by_juris.plot(kind='bar', title='Suppression rate by jurisdiction (top 15)', figsize=(10, 4))
plt.tight_layout()
plt.show()

## 2. Target distribution per category

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, cat in zip(axes, ['all_drugs', 'all_opioids', 'all_stimulants']):
    vals = train.loc[train['overdose_category'] == cat, 'rate_per_10000_ed_visits'].dropna()
    ax.hist(vals, bins=40, edgecolor='white')
    ax.set_title(cat)
    ax.set_xlabel('rate per 10k ED visits')
    print(f"{cat}: n={len(vals)}  median={vals.median():.2f}  p95={vals.quantile(.95):.2f}")
plt.tight_layout()
plt.show()

## 3. Period_id structure — can we decode the opaque IDs?

In [ ]:
periods = train['period_id'].unique()
print(f'{len(periods)} unique train period_ids:', sorted(periods))
val_periods = val['period_id'].unique()
print(f'{len(val_periods)} unique val period_ids:', sorted(val_periods))
print('\nOverlap:', set(periods) & set(val_periods))

In [ ]:
# Inferred temporal order via mean temperature proxy
rank_check = train.groupby(['period_id', 'period_rank'])['temp_avg_f'].mean().reset_index()
rank_check = rank_check.sort_values('period_rank')
print(rank_check[['period_id', 'period_rank', 'temp_avg_f']].to_string(index=False))

## 4. Google Trends vs target correlation

In [ ]:
gtrends_cols = ['gtrends_overdose', 'gtrends_fentanyl', 'gtrends_naloxone',
                'gtrends_opioid', 'gtrends_methamphetamine']

for cat in ['all_drugs', 'all_opioids', 'all_stimulants']:
    subset = train[train['overdose_category'] == cat].dropna(subset=['rate_per_10000_ed_visits'])
    corr = subset[gtrends_cols + ['rate_per_10000_ed_visits']].corr()['rate_per_10000_ed_visits'][gtrends_cols]
    print(f'\n{cat} — Pearson r vs target:')
    print(corr.to_string())

## 5. Sample DoH release text

In [ ]:
nonempty = train[train['state_doh_release'].fillna('').str.strip() != '']['state_doh_release']
print(f'Non-empty DoH releases: {len(nonempty)} / {len(train)} rows\n')
for i, text in enumerate(nonempty.sample(3, random_state=42)):
    print(f'--- Sample {i+1} ---')
    print(text[:600])
    print()

## 6. MAT density images — sample display

In [ ]:
from pathlib import Path
from PIL import Image

img_dir = Path('..') / 'train' / 'images' / 'mat_density'
imgs = sorted(img_dir.glob('*.png'))
print(f'{len(imgs)} MAT density images found')

if imgs:
    fig, axes = plt.subplots(1, min(3, len(imgs)), figsize=(12, 4))
    if len(imgs) == 1:
        axes = [axes]
    for ax, p in zip(axes, imgs[:3]):
        ax.imshow(Image.open(p))
        ax.set_title(p.stem, fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 7. Missing-value heatmap

In [ ]:
num_cols = ['unemployment_rate','labor_force','temp_avg_f','precip_in',
            'gtrends_overdose','gtrends_fentanyl','gtrends_naloxone',
            'gtrends_opioid','gtrends_methamphetamine']
miss = train[num_cols].isna().mean().sort_values(ascending=False)
print('Missing rate per covariate:')
print(miss.to_string())